In [ ]:
%%html
<div>Teachable Machine Image Model</div>
<button type="button" onclick="init()">Start</button>
<div id="webcam-container"></div>
<div id="label-container"></div>
<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs@latest/dist/tf.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/@teachablemachine/image@latest/dist/teachablemachine-image.min.js"></script>
<script type="text/javascript">
    // More API functions here:
    // https://github.com/googlecreativelab/teachablemachine-community/tree/master/libraries/image

    // the link to your model provided by Teachable Machine export panel
    const URL = "https://teachablemachine.withgoogle.com/models/vL48UjKMs/";

    let model, webcam, labelContainer, maxPredictions;

    // Load the image model and setup the webcam
    async function init() {
        const modelURL = URL + "model.json";
        const metadataURL = URL + "metadata.json";

        // load the model and metadata
        // Refer to tmImage.loadFromFiles() in the API to support files from a file picker
        // or files from your local hard drive
        // Note: the pose library adds "tmImage" object to your window (window.tmImage)
        model = await tmImage.load(modelURL, metadataURL);
        maxPredictions = model.getTotalClasses();

        // Convenience function to setup a webcam
        const flip = true; // whether to flip the webcam
        webcam = new tmImage.Webcam(200, 200, flip); // width, height, flip
        await webcam.setup(); // request access to the webcam
        await webcam.play();
        window.requestAnimationFrame(loop);

        // append elements to the DOM
        document.getElementById("webcam-container").appendChild(webcam.canvas);
        labelContainer = document.getElementById("label-container");
        for (let i = 0; i < maxPredictions; i++) { // and class labels
            labelContainer.appendChild(document.createElement("div"));
        }
    }

    async function loop() {
        webcam.update(); // update the webcam frame
        await predict();
        window.requestAnimationFrame(loop);
    }

    // run the webcam image through the image model
    async function predict() {
        // predict can take in an image, video or canvas html element
        const prediction = await model.predict(webcam.canvas);
        for (let i = 0; i < maxPredictions; i++) {
            const classPrediction =
                prediction[i].className + ": " + prediction[i].probability.toFixed(2);
            labelContainer.childNodes[i].innerHTML = classPrediction;
        }
    }
</script>

In [ ]:
%%html
<div>Teachable Machine Image Model - Upload de Imagem Local</div>
<br>
<!-- Input para o usuário fazer upload da imagem -->
<input type="file" id="image-selector" accept="image/*" onchange="handleImageUpload(event)" disabled>
<br><br>
<div id="image-container"></div>
<div id="label-container"></div>

<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs@latest/dist/tf.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/@teachablemachine/image@latest/dist/teachablemachine-image.min.js"></script>
<script type="text/javascript">
    const URL = "https://teachablemachine.withgoogle.com/models/vL48UjKMs/";
    let model, labelContainer, maxPredictions;

    // Inicializa o modelo assim que a página/célula carrega
    async function initModel() {
        const modelURL = URL + "model.json";
        const metadataURL = URL + "metadata.json";

        // Carrega o modelo
        model = await tmImage.load(modelURL, metadataURL);
        maxPredictions = model.getTotalClasses();

        labelContainer = document.getElementById("label-container");
        labelContainer.innerHTML = ""; // Limpa labels anteriores
        for (let i = 0; i < maxPredictions; i++) {
            labelContainer.appendChild(document.createElement("div"));
        }

        // Ativa o input de arquivo após carregar o modelo
        document.getElementById("image-selector").disabled = false;
        console.log("Modelo carregado com sucesso! Já pode selecionar uma imagem.");
    }

    // Função que lida com a imagem selecionada pelo usuário
    function handleImageUpload(event) {
        const file = event.target.files[0];
        if (!file) return;

        const reader = new FileReader();
        reader.onload = function(e) {
            const imgContainer = document.getElementById("image-container");
            imgContainer.innerHTML = ""; // Limpa imagem anterior

            // Cria o elemento de imagem na tela
            const img = document.createElement("img");
            img.src = e.target.result;
            img.width = 200; // Define largura para visualização
            img.onload = async function() {
                // Executa a predição assim que a imagem for totalmente carregada
                await predict(img);
            };
            imgContainer.appendChild(img);
        };
        reader.readAsDataURL(file);
    }

    // Executa a previsão na imagem carregada
    async function predict(imageElement) {
        const prediction = await model.predict(imageElement);
        for (let i = 0; i < maxPredictions; i++) {
            const classPrediction =
                prediction[i].className + ": " + (prediction[i].probability * 100).toFixed(2) + "%";
            labelContainer.childNodes[i].innerHTML = classPrediction;
        }
    }

    // Executa o carregamento inicial do modelo
    initModel();
</script>

In [ ]:
%%html
<div>
  <h3>Teachable Machine - Webcam, Upload e Histórico em CSV</h3>

  <!-- Controles da Webcam -->
  <button type="button" onclick="startWebcam()">Iniciar Webcam</button>
  <button type="button" onclick="stopWebcam()">Parar Webcam</button>
  <br><br>

  <!-- Controle de Upload de Imagem -->
  <label>Ou faça upload de uma imagem: </label>
  <input type="file" id="image-selector" accept="image/*" onchange="handleImageUpload(event)" disabled>
  <br><br>

  <!-- Área de exibição de mídia (Webcam ou Imagem Estática) -->
  <div id="media-container" style="min-height: 200px; border: 1px dashed #ccc; display: inline-block; padding: 10px;">
    <p style="color: #666;">A mídia ativa aparecerá aqui</p>
  </div>

  <!-- Resultados Atuais -->
  <div id="label-container" style="margin-top: 15px; font-weight: bold;"></div>
  <br>

  <!-- Controle do Histórico -->
  <button type="button" onclick="downloadCSV()">Exportar Histórico para CSV</button>
  <span id="history-counter" style="margin-left: 10px; color: #555;">Predições gravadas: 0</span>
</div>

<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs@latest/dist/tf.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/@teachablemachine/image@latest/dist/teachablemachine-image.min.js"></script>
<script type="text/javascript">
    const URL = "https://teachablemachine.withgoogle.com/models/vL48UjKMs/";
    let model, webcam, labelContainer, maxPredictions;
    let isWebcamActive = false;
    let animationFrameId = null;

    // Array para armazenar o histórico de predições
    let predictionHistory = [];
    let currentSource = "Nenhum";

    // Inicializa o modelo assim que a célula é executada
    async function init() {
        const modelURL = URL + "model.json";
        const metadataURL = URL + "metadata.json";

        model = await tmImage.load(modelURL, metadataURL);
        maxPredictions = model.getTotalClasses();

        labelContainer = document.getElementById("label-container");
        labelContainer.innerHTML = "";
        for (let i = 0; i < maxPredictions; i++) {
            labelContainer.appendChild(document.createElement("div"));
        }

        // Habilita o campo de upload de imagem após carregar o modelo
        document.getElementById("image-selector").disabled = false;
    }

    // Limpa a área de mídia antes de carregar um novo conteúdo
    function clearMediaContainer() {
        stopWebcam();
        const container = document.getElementById("media-container");
        container.innerHTML = "";
    }

    // --- FLUXO DA WEBCAM ---
    async function startWebcam() {
        clearMediaContainer();

        const container = document.getElementById("media-container");
        const flip = true;
        webcam = new tmImage.Webcam(200, 200, flip);

        await webcam.setup(); // Solicita acesso à câmera
        await webcam.play();

        container.appendChild(webcam.canvas);
        isWebcamActive = true;
        currentSource = "Webcam";
        loop();
    }

    function stopWebcam() {
        isWebcamActive = false;
        if (animationFrameId) {
            cancelAnimationFrame(animationFrameId);
            animationFrameId = null;
        }
        if (webcam) {
            webcam.stop();
        }
    }

    async function loop() {
        if (!isWebcamActive) return;
        webcam.update();
        await predict(webcam.canvas);
        animationFrameId = window.requestAnimationFrame(loop);
    }

    // --- FLUXO DE UPLOAD DE IMAGEM ---
    function handleImageUpload(event) {
        const file = event.target.files[0];
        if (!file) return;

        clearMediaContainer();
        const container = document.getElementById("media-container");
        currentSource = "Upload: " + file.name;

        const reader = new FileReader();
        reader.onload = function(e) {
            const img = document.createElement("img");
            img.src = e.target.result;
            img.width = 200;
            img.onload = async function() {
                await predict(img);
            };
            container.appendChild(img);
        };
        reader.readAsDataURL(file);
    }

    // --- PREVISÃO E REGISTRO NO HISTÓRICO ---
    async function predict(inputElement) {
        const prediction = await model.predict(inputElement);

        // Cria um objeto para a predição atual
        let record = {
            timestamp: new Date().toISOString(),
            origem: currentSource
        };

        for (let i = 0; i < maxPredictions; i++) {
            const className = prediction[i].className;
            const probability = (prediction[i].probability * 100).toFixed(2);

            // Atualiza o texto na interface gráfica
            labelContainer.childNodes[i].innerHTML = className + ": " + probability + "%";

            // Adiciona a classe e a porcentagem ao registro
            record[className] = probability + "%";
        }

        // Salva no histórico (evita encher a memória excessivamente na webcam limitando a taxa se necessário,
        // mas para este exemplo, gravaremos todos os frames analisados)
        predictionHistory.push(record);
        document.getElementById("history-counter").innerText = "Predições gravadas: " + predictionHistory.length;
    }

    // --- EXPORTAR HISTÓRICO EM CSV ---
    function downloadCSV() {
        if (predictionHistory.length === 0) {
            alert("Nenhuma predição foi gravada ainda!");
            return;
        }

        // Define os cabeçalhos do CSV com base nas chaves do primeiro registro
        const headers = Object.keys(predictionHistory[0]);
        let csvRows = [headers.join(",")];

        // Mapeia cada registro para uma linha do CSV
        for (const row of predictionHistory) {
            const values = headers.map(header => {
                const escaped = ('' + row[header]).replace(/"/g, '\"');
                return `"${escaped}"`;
            });
            csvRows.push(values.join(","));
        }

        // Cria o Blob e dispara o download no navegador
        const csvContent = "\ufeff" + csvRows.join("\n"); // \ufeff para compatibilidade de acentos no Excel
        const blob = new Blob([csvContent], { type: 'text/csv;charset=utf-8;' });
        const url = URL.createObjectURL(blob);
        const link = document.createElement("a");

        link.setAttribute("href", url);
        link.setAttribute("download", "historico_predicoes.csv");
        link.style.visibility = 'hidden';
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
    }

    // Inicia o carregamento do modelo
    init();
</script>